## Load useful libraries

In [1]:
import os
import shutil
from pathlib import Path
import pandas as pd
import numpy as np

import pprint as pp

In [44]:
from biological_graph_database.config import ftp_root_ncbi
from biological_graph_database.tools import download_file
from biological_graph_database.tools import compute_file_md5, read_md5_file
from biological_graph_database.tools import Neo4jInterface

## User settings

In [3]:
remote_directory = '/pub/taxonomy'
email_address = os.environ['EMAIL_ADDRESS']
directory_download_to = os.environ['BIOLOGICAL_GRAPH_DATABASE_HOME'] + '/data/taxonomy'

## Create the data directory for the taxonomy download

In [ ]:
shutil.rmtree(directory_download_to, ignore_errors = True)
Path(directory_download_to).mkdir()

## Download the taxonomy data

In [ ]:
file_to_process = download_file('http://' + ftp_root_ncbi + '/' + remote_directory + '/taxdump.tar.gz', directory_download_to)
file_to_process_md5 = download_file('http://' + ftp_root_ncbi + '/' + remote_directory + '/taxdump.tar.gz.md5', directory_download_to)

## Check whether download succeeded

In [ ]:
assert compute_file_md5(file_to_process) == read_md5_file(file_to_process_md5)

## Decompress the downloaded file

In [ ]:
directory_working = os.getcwd()
os.chdir(directory_download_to)
os.system('gunzip taxdump.tar.gz')
os.system('tar -xf taxdump.tar')
os.system('gzip taxdump.tar')
os.chdir(directory_working)

In [4]:
import re

with open(directory_download_to + '/readme.txt') as f:
    list_lines = [x.strip() for x in f.readlines()]

new_list_lines = []
for line in list_lines:
    new_line = line
    
    #new_line = line.split('\t')[0].split('--')[0].strip()
    #new_line = line.split('\t')[0].strip()
    
    new_line = re.sub(' +', ' ', new_line)
    new_list_lines.append(new_line)
list_lines = new_list_lines



dict_results = {}
current_dmp_filename = None
for line in list_lines:
    if line.find('.dmp') >= 0 and line.find('--') == -1:

        current_dmp_filename = line.split('.dmp')[0].strip() + '.dmp'
        
        if current_dmp_filename.find('(') >= 0:
            thing = current_dmp_filename.split('(')
            for item in thing:
                if '.dmp' in item:
                    current_dmp_filename = item.strip()

        current_dmp_filename = current_dmp_filename.replace(')', '')
        current_dmp_filename = current_dmp_filename.strip()
        
        dict_results[current_dmp_filename] = []
    else:
        if line.strip() != '':
            to_add = line

            if to_add.find('--') >= 0 or to_add.strip().find('comments') == 0:
                to_add = to_add.split('\t')[0]

                to_add = to_add.split('--')[0]
            
                to_add = re.sub(' +', ' ', to_add)
            
                to_add = to_add.replace('(0 or 1)', '')
                to_add = to_add.replace('(1 or 0)', '')

                to_add = to_add.strip()
                to_add = to_add.replace(' ', '_')


                
                to_add = to_add.strip()
                if current_dmp_filename != None and to_add != '':
                    dict_results[current_dmp_filename].append(to_add)


del(dict_results['*.dmp'])

pp.pprint(dict_results)

{'citations.dmp': ['cit_id',
                   'cit_key',
                   'pubmed_id',
                   'medline_id',
                   'url',
                   'text',
                   'taxid_list'],
 'delnodes.dmp': ['tax_id'],
 'division.dmp': ['division_id', 'division_cde', 'division_name', 'comments'],
 'gencode.dmp': ['genetic_code_id', 'abbreviation', 'name', 'cde', 'starts'],
 'images.dmp': ['image_id',
                'image_key',
                'url',
                'license',
                'attribution',
                'source',
                'properties',
                'taxid_list'],
 'merged.dmp': ['old_tax_id', 'new_tax_id'],
 'names.dmp': ['tax_id', 'name_txt', 'unique_name', 'name_class'],
 'nodes.dmp': ['tax_id',
               'parent_tax_id',
               'rank',
               'embl_code',
               'division_id',
               'inherited_div_flag',
               'genetic_code_id',
               'inherited_GC_flag',
               'mitoc

In [5]:
def load_file_to_df(filename, columns):
    df = pd.read_csv(filename, sep = '\t', header = None)
    df = df.drop(columns = [x for x in range(0, len(df.columns)) if x % 2 == 1])
    df.columns = columns
    return df

In [6]:
df_nodes = load_file_to_df(directory_download_to + '/nodes.dmp', dict_results['nodes.dmp'])
df_nodes.head(3)

,tax_id,parent_tax_id,rank,embl_code,division_id,inherited_div_flag,genetic_code_id,inherited_GC_flag,mitochondrial_genetic_code_id,inherited_MGC_flag,GenBank_hidden_flag,hidden_subtree_root_flag,comments
0,1,1,no rank,NaN,8,0,1,0,0,0,0,0,NaN
1,2,131567,domain,NaN,0,0,11,0,0,0,0,0,NaN
2,6,335928,genus,NaN,0,1,11,1,0,1,0,0,code compliant


In [7]:
assert len(df_nodes.index) == len(df_nodes['tax_id'].unique())

In [8]:
set_tax_ids = set(list([np.int64(x) for x in df_nodes['tax_id'].unique()]))
set_parent_tax_ids = set(list([np.int64(x) for x in df_nodes['parent_tax_id'].unique()]))

In [9]:
sorted(list(set_tax_ids))[0:5]

[np.int64(1), np.int64(2), np.int64(6), np.int64(7), np.int64(9)]

In [10]:
sorted(list(set_parent_tax_ids))[0:5]

[np.int64(1), np.int64(2), np.int64(6), np.int64(7), np.int64(9)]

In [11]:
df_delnodes = load_file_to_df(directory_download_to + '/delnodes.dmp', dict_results['delnodes.dmp'])
df_delnodes.head(3)

,tax_id
0,3712593
1,3712592
2,3712542


In [12]:
set_deleted_tax_ids = set(list([np.int64(x) for x in df_delnodes['tax_id'].unique()]))

In [13]:
sorted(list(set_deleted_tax_ids))[0:5]

[np.int64(3), np.int64(4), np.int64(5), np.int64(8), np.int64(15)]

In [14]:
df_merged = load_file_to_df(directory_download_to + '/merged.dmp', dict_results['merged.dmp'])
df_merged.head(3)

,old_tax_id,new_tax_id
0,12,74109
1,30,29
2,36,184914


In [15]:
set_old_tax_ids = set(list([np.int64(x) for x in df_merged['old_tax_id'].unique()]))
set_new_tax_ids = set(list([np.int64(x) for x in df_merged['new_tax_id'].unique()]))

In [16]:
sorted(list(set_old_tax_ids))[0:5]

[np.int64(12), np.int64(30), np.int64(36), np.int64(37), np.int64(46)]

In [17]:
sorted(list(set_new_tax_ids))[0:5]

[np.int64(7), np.int64(9), np.int64(14), np.int64(17), np.int64(23)]

In [18]:
df_names = load_file_to_df(directory_download_to + '/names.dmp', dict_results['names.dmp'])
df_names.head(5)

,tax_id,name_txt,unique_name,name_class
0,1,all,NaN,synonym
1,1,root,NaN,scientific name
2,2,Bacteria,Bacteria <bacteria>,scientific name
3,2,bacteria,bacteria <blast name>,blast name
4,2,bacteria,bacteria <genbank common name>,genbank common name


In [19]:
set_names_tax_ids = set(list([np.int64(x) for x in df_names['tax_id'].unique()]))

In [20]:
sorted(list(set_names_tax_ids))[0:5]

[np.int64(1), np.int64(2), np.int64(6), np.int64(7), np.int64(9)]

In [21]:
import re

def process_taxid_list(df, keyname = 'taxid_list'):
    df_temp = df[[keyname]].copy().dropna()
    list_taxid_list = [str(x).strip() for x in list(df_temp[keyname].values)]
    list_taxid_list = [re.sub(' +', ' ', x) for x in list_taxid_list]
    list_taxid_list = [x.split(' ') for x in list_taxid_list]
    list_taxid_list = [x for x in list_taxid_list if len(x) > 0]
    
    new_list = []
    for item in list_taxid_list:
        new_list.append([np.int64(x) for x in item])
    
    return new_list

def convert_to_set_containing_all_items(df, keyname = 'taxid_list'):
    list_keyname_records = process_taxid_list(df, keyname)
    list_keynames = []
    for item in list_keyname_records:
        list_keynames.extend(item)
    set_keynames = set(list_keynames)
    return set_keynames   

In [22]:
df_images = load_file_to_df(directory_download_to + '/images.dmp', dict_results['images.dmp'])
df_images.head(3)

,image_id,image_key,url,license,attribution,source,properties,taxid_list
0,64364,image:Candidatus Aciduliprofundum boonei,http://www.ncbi.nlm.nih.gov/Taxonomy/taxi/imag...,Public Domain (https://creativecommons.org/pub...,NaN,Wikimedia Commons,NaN,379547
1,64365,image:Cyanophora paradoxa,http://www.ncbi.nlm.nih.gov/Taxonomy/taxi/imag...,CC BY-SA 3.0 (https://creativecommons.org/lice...,Wolfgang Bettighofer,Wikimedia Commons,NaN,2762
2,64366,image:Emiliania huxleyi,http://www.ncbi.nlm.nih.gov/Taxonomy/taxi/imag...,CC BY-SA 4.0 (https://creativecommons.org/lice...,Dr. Jeremy Young,Wikimedia Commons,NaN,2903


In [23]:
set_images_tax_ids = convert_to_set_containing_all_items(df_images)

In [24]:
sorted(list(set_images_tax_ids))[0:5] 

[np.int64(160), np.int64(196), np.int64(197), np.int64(263), np.int64(271)]

In [25]:
def fix_file(filename, sep = '\t'):
    with open(filename) as f:
        lines = [x.strip() for x in f.readlines()]

    count_columns = 0
    for line in lines:
        count_columns_this_line = len(line.split(sep))
        if count_columns_this_line > count_columns:
            count_columns = count_columns_this_line
            print(line)
            print()

    list_new_lines = []
    for line in lines:
        count_columns_this_line = len(line.split(sep))
        new_line = line + (sep * (count_columns - count_columns_this_line))
        list_new_lines.append(new_line)
        
    with open(filename + '_fixed', 'w') as f:
        f.write('\n'.join(list_new_lines) + '\n')

In [26]:
fix_file(directory_download_to + '/citations.dmp')

5	|	The domestic cat: perspective on the nature and diversity of cats.	|	0	|	8603894	|	 	|		|	9685 	|

36396	|	The Plant List - Polygala	|	0	|	0	|	Gymnadenia rhellicani	Nigritella rhellicani	|		|	4275 1384028 1384029 	|



In [27]:
# crude
columns_to_use = dict_results['citations.dmp'].copy()
columns_to_use.append('unknown_column')

df_citations = load_file_to_df(directory_download_to + '/citations.dmp_fixed', columns_to_use)
df_citations.head(5)

,cit_id,cit_key,pubmed_id,medline_id,url,text,taxid_list,unknown_column
0,5,The domestic cat: perspective on the nature an...,0,8603894,,NaN,9685,NaN
1,8,Yabuuchi E et al. (1990),0,2111872,https://doi.org/10.1111/j.1348-0421.1990.tb009...,"Yabuuchi, E., Yano, I., Oyaizu, H., Hashimoto,...",13687 13688 13689 28212 28213,NaN
2,9,Dennis PJ et al. (1993),0,8494743,NaN,"Dennis, P.J., Brenner, D.J., Thacker, W.L., Wa...",45065 45068 45070 45072 45076,NaN
3,54,Rainey FA et al. (1995),0,7857805,NaN,"Rainey, F.A., Klatte, S., Kroppenstedt, R.M., ...",37914 37915,NaN
4,55,Ludwig W et al. (1992),0,1371060,NaN,"Ludwig, W., Kirchhof, G., Weizenegger, M., and...",1657,NaN


In [28]:
# crude
df_citations = df_citations[df_citations['taxid_list'] != '|'].copy()
df_citations.head(3)

,cit_id,cit_key,pubmed_id,medline_id,url,text,taxid_list,unknown_column
0,5,The domestic cat: perspective on the nature an...,0,8603894,,NaN,9685,NaN
1,8,Yabuuchi E et al. (1990),0,2111872,https://doi.org/10.1111/j.1348-0421.1990.tb009...,"Yabuuchi, E., Yano, I., Oyaizu, H., Hashimoto,...",13687 13688 13689 28212 28213,NaN
2,9,Dennis PJ et al. (1993),0,8494743,NaN,"Dennis, P.J., Brenner, D.J., Thacker, W.L., Wa...",45065 45068 45070 45072 45076,NaN


In [29]:
set_citations_tax_ids = convert_to_set_containing_all_items(df_citations)

In [30]:
sorted(list(set_citations_tax_ids))[0:5] 

[np.int64(1), np.int64(2), np.int64(6), np.int64(7), np.int64(9)]

In [ ]:
set_parent_tax_ids.issubset(set_tax_ids)

In [ ]:
set_deleted_tax_ids.issubset(set_tax_ids)

In [ ]:
set_old_tax_ids.issubset(set_tax_ids)

In [ ]:
set_new_tax_ids.issubset(set_tax_ids)

In [ ]:
set_tax_ids.issubset(set_names_tax_ids)

In [ ]:
set_names_tax_ids.issubset(set_tax_ids)

In [ ]:
set_deleted_tax_ids.issubset(set_names_tax_ids)

In [ ]:
set_old_tax_ids.issubset(set_names_tax_ids)

In [ ]:
set_new_tax_ids.issubset(set_names_tax_ids)

In [31]:
set_all_tax_ids = (
    set_tax_ids
    .union(set_names_tax_ids)
    .union(set_deleted_tax_ids)
    .union(set_old_tax_ids)
    .union(set_new_tax_ids)
    .union(set_images_tax_ids)
    .union(set_citations_tax_ids)
)

In [32]:
#sorted(list(set_all_tax_ids))[0:10]

In [33]:
df_all_tax_id_nodes = pd.DataFrame({'tax_id' : sorted(list(set_all_tax_ids))})
df_all_tax_id_nodes.head(3)

,tax_id
0,1
1,2
2,3


In [34]:
df_tax_id_has_parent_tax_id_relationship = df_nodes[['tax_id', 'parent_tax_id']].copy()

In [35]:
df_tax_id_has_parent_tax_id_relationship.head(3)

,tax_id,parent_tax_id
0,1,1
1,2,131567
2,6,335928


In [36]:
list_taxonomy_ranks = sorted([x.lower() for x in list(df_nodes['rank'].unique())])
df_taxonomy_ranks = pd.DataFrame({'taxonomy_ranks' : list_taxonomy_ranks})

In [37]:
df_taxonomy_ranks.head(3)

,taxonomy_ranks
0,acellular root
1,biotype
2,cellular root


In [38]:
df_tax_id_has_rank_relationship = df_nodes[['tax_id', 'rank']].copy()
df_tax_id_has_rank_relationship['rank'] = [x.lower() for x in df_tax_id_has_rank_relationship['rank']]

In [39]:
df_tax_id_has_rank_relationship.head(3)

,tax_id,rank
0,1,no rank
1,2,domain
2,6,genus


In [45]:
nj = Neo4jInterface('aoeuI823')

In [47]:
def batch_tax_id(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:TAXONOMY {tax_id: row.tax_id})
    """
    tx.run(query, batch=batch)

In [ ]:
nj.batch_it(batch_tax_id, df_all_tax_id_nodes.sort_values(by = ['tax_id']).to_dict(orient = 'records'))

In [ ]:
def batch_tax_rank(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:TAXONOMY_RANK {name: row.taxonomy_ranks})
    """
    tx.run(query, batch=batch)

In [ ]:
df_taxonomy_ranks.sort_values(by = ['taxonomy_ranks']).to_dict(orient = 'records')[0:5]